## Libraries

In [29]:
from langchain_google_genai import GoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

print("libraries loaded")

import os
import re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
print("env loaded")

libraries loaded
env loaded


## Model

In [30]:
model = GoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0.7)

print("model ready")

model ready


## Text Preprocessing 

In [31]:
def preprocess_document(text) -> str:
    text = re.sub(r'Page \d+\s*(?:of|\/)\s*\d+', '', text) #page numbers

    text = re.sub(r'—{2,}|_{3,}', '', text) # extra dashes/lines
    
    text = re.sub(r'\s+', ' ', text).strip() # whitespace
    return text

In [32]:
def preprocess_query(query) -> str:
    # whitespace on user query
    return " ".join(query.split()).strip()

## Prompt Template

In [33]:
Template = """
You are a helpful assistant that answer questions strictly based on the provided document context.

rules:
- Answer from the context below. Do NOT use outside knowledge.
- If the answer not within context, clearly say so
- Be concise but complete. Cite the relevant part when useful.

Context:
{context}

Question: {question}

Answer:
"""

prompt = PromptTemplate(
    template=Template,
    input_variables=["context", "question"]
)

print("Prompt template ready")

Prompt template ready


## Loading the pdf

In [35]:
pdf_path = "PDF/Distributed_Database_Architectures.pdf"

if not Path(pdf_path).exists():
    raise FileNotFoundError(f"PDF not found: '{pdf_path}'")

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"loaded {len(documents)} pages")

# applying pre-processing
for doc in documents:
    doc.page_content = preprocess_document(doc.page_content)

loaded 1 pages


## Splitting chunks

In [36]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(documents)

print(f"split to {len(chunks)} chunks")

split to 1 chunks


## Vector Store

In [37]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Derive a cache folder name from the PDF filename
index_path = Path(pdf_path).stem + "_faiss_index"

if Path(index_path).exists():
    vector_store = FAISS.load_local(index_path, embeddings, allow_dangerous_deserialization=True)
else:
    vector_store = FAISS.from_documents(chunks, embeddings)
    vector_store.save_local(index_path)
    print(f"saved to '{index_path}'")

print("Vector store ready to use")

saved to 'Distributed_Database_Architectures_faiss_index'
Vector store ready to use


## QA chain

In [38]:
qa_chain = RetrievalQA.from_chain_type(
    llm=model,
    chain_type="stuff",
    retriever=vector_store.as_retriever(search_kwargs={"k": 3}),
    chain_type_kwargs={"prompt": prompt}
)

print("QA chain redy")

QA chain redy


## Playground

In [39]:
print("\nPDF Analyzer")
print("=" * 20)

while True:
    raw_query = input("\n Ready when you are, enter your question (or type 'quit' to exit): ")
    
    if raw_query.lower() == "quit":
        print("bye bye")
        break
    
    # Preprocess query minimally
    query = preprocess_query(raw_query)
    print(f"Question:\n{query}")


    try:
        result = qa_chain.invoke({"query": query})
        print(f"\nAnswer:\n{result['result']}")
    except Exception as E:
        print(f"Error {E}")


PDF Analyzer
Question:
what is this pdf talking about

Answer:
This document is talking about Distributed Database Architectures, specifically detailing the Shared-Nothing Architecture and the Shared-Disk Architecture, including their characteristics, examples, pros, and cons.
bye bye
